In [ ]:
!pip install -r requirements.txt -q

In [1]:
import numpy as np
import pandas as pd
import json
import csv
from tqdm import tqdm

## Load Datasets
1. [CMU book summaries](https://www.cs.cmu.edu/~dbamman/booksummaries.html)
2. [Open Library works dump](https://openlibrary.org/developers/dumps)

In [3]:
# !wget https://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz
# !tar -xvzf booksummaries.tar.gz

--2026-03-27 19:32:28--  https://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz
Resolving www.cs.cmu.edu (www.cs.cmu.edu)... 128.2.42.95
Connecting to www.cs.cmu.edu (www.cs.cmu.edu)|128.2.42.95|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16795330 (16M) [application/x-gzip]
Saving to: ‘booksummaries.tar.gz.1’

booksummaries.tar.g 100%[===================>]  16.02M  8.46MB/s    in 1.9s    

2026-03-27 19:32:30 (8.46 MB/s) - ‘booksummaries.tar.gz.1’ saved [16795330/16795330]

booksummaries/
booksummaries/README
booksummaries/booksummaries.txt


In [ ]:
# !wget https://openlibrary.org/data/ol_dump_works_latest.txt.gz
# !gzip -d ol_dump_works_latest.txt.gz

## Clean Data from CMU Book Summaries Dataset

If the workspace contains `booksummaries_processed.csv`, run the following cell. Skip the remaining section.

In [9]:
df = pd.read_csv('booksummaries_processed.csv')
df['genres_list'] = df['genres_list'].apply(json.loads)
print(df.head())

   wiki_id freebase_id                                      title  \
0      620     /m/0hhy                                Animal Farm   
1      843     /m/0k36                         A Clockwork Orange   
2      986     /m/0ldx                                 The Plague   
3     1756     /m/0sww  An Enquiry Concerning Human Understanding   
4     2080     /m/0wkt                       A Fire Upon the Deep   

            author    pub_date  \
0    George Orwell  1945-08-17   
1  Anthony Burgess        1962   
2     Albert Camus        1947   
3       David Hume         NaN   
4     Vernor Vinge         NaN   

                                              genres  \
0  {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...   
1  {"/m/06n90": "Science Fiction", "/m/0l67h": "N...   
2  {"/m/02m4t": "Existentialism", "/m/02xlf": "Fi...   
3                                                NaN   
4  {"/m/03lrw": "Hard science fiction", "/m/06n90...   

                                           

In [4]:
cols = ['wiki_id', 'freebase_id', 'title', 'author', 'pub_date', 'genres', 'summary']
df = pd.read_csv('booksummaries/booksummaries.txt', sep='\t', header=None, names=cols)
print(df.head())

df.info()
# Drop books with no summary
df = df[df['summary'].notna() & (df['summary'].str.split().str.len() > 50)]
df.info()

   wiki_id freebase_id                                      title  \
0      620     /m/0hhy                                Animal Farm   
1      843     /m/0k36                         A Clockwork Orange   
2      986     /m/0ldx                                 The Plague   
3     1756     /m/0sww  An Enquiry Concerning Human Understanding   
4     2080     /m/0wkt                       A Fire Upon the Deep   

            author    pub_date  \
0    George Orwell  1945-08-17   
1  Anthony Burgess        1962   
2     Albert Camus        1947   
3       David Hume         NaN   
4     Vernor Vinge         NaN   

                                              genres  \
0  {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...   
1  {"/m/06n90": "Science Fiction", "/m/0l67h": "N...   
2  {"/m/02m4t": "Existentialism", "/m/02xlf": "Fi...   
3                                                NaN   
4  {"/m/03lrw": "Hard science fiction", "/m/06n90...   

                                           

In [10]:
def parse_genres(g):
    try:
        return list(json.loads(g).values())
    except:
        return []

df['genres_list'] = df['genres'].apply(parse_genres)
print(df['genres_list'])
df = df.reset_index(drop=True)
print(f"Books after cleaning: {len(df)}")

0        [Roman à clef, Satire, Children's literature, ...
1        [Science Fiction, Novella, Speculative fiction...
2        [Existentialism, Fiction, Absurdist fiction, N...
3                                                       []
4        [Hard science fiction, Science Fiction, Specul...
                               ...                        
15465                                                   []
15466                                                   []
15467                                  [Thriller, Fiction]
15468                                      [Autobiography]
15469              [Epistolary novel, Speculative fiction]
Name: genres_list, Length: 15470, dtype: object
Books after cleaning: 15470


In [11]:
import nltk, re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download(['stopwords','wordnet','punkt'])
stop = set(stopwords.words('english'))
lem = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [lem.lemmatize(t) for t in tokens if t not in stop]
    return ' '.join(tokens)

[nltk_data] Downloading package stopwords to /home/irisx2/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/irisx2/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/irisx2/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [7]:
df['clean_summary'] = df['summary'].apply(preprocess)

df['combined'] = (df['clean_summary'] + ' ' +
                  df['author'].fillna('') + ' ' +
                  df['genres_list'].apply(lambda g: ' '.join(g)))

print(df.head())
df['genres_list'] = df['genres_list'].apply(json.dumps)
df.to_csv('booksummaries_processed.csv', index=False)

[nltk_data] Downloading package stopwords to /home/irisx2/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/irisx2/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/irisx2/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


   wiki_id freebase_id                                      title  \
0      620     /m/0hhy                                Animal Farm   
1      843     /m/0k36                         A Clockwork Orange   
2      986     /m/0ldx                                 The Plague   
3     1756     /m/0sww  An Enquiry Concerning Human Understanding   
4     2080     /m/0wkt                       A Fire Upon the Deep   

            author    pub_date  \
0    George Orwell  1945-08-17   
1  Anthony Burgess        1962   
2     Albert Camus        1947   
3       David Hume         NaN   
4     Vernor Vinge         NaN   

                                              genres  \
0  {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...   
1  {"/m/06n90": "Science Fiction", "/m/0l67h": "N...   
2  {"/m/02m4t": "Existentialism", "/m/02xlf": "Fi...   
3                                                NaN   
4  {"/m/03lrw": "Hard science fiction", "/m/06n90...   

                                           

## TF-IDF Computation

If `tfidf_matrix.pkl` and `tfidf_vectorizer.pkl` already exist in the workspace, run the following cell and jump to the [First Recommendations](#First-Recommendations) section.

In [12]:
# load, if already exists
import joblib

tfidf_matrix = joblib.load('tfidf_matrix.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
!pip install python-dotenv -q

In [11]:
import os
from dotenv import load_dotenv

# have HF_TOKEN in .env file for faster loading
load_dotenv()

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['combined'])

corpus = [text.split() for text in df['clean_summary']]
bm25 = BM25Okapi(corpus)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = SentenceTransformer('all-MiniLM-L6-v2')
batch_size = 512 if device == 'cuda' else 64
embeddings = model.encode(
    df['clean_summary'].tolist(),
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=device
)

joblib.dump(tfidf_matrix, 'tfidf_matrix.pkl')

Batches: 100%|██████████| 242/242 [15:54<00:00,  3.94s/it]


['tfidf_matrix.pkl']

In [12]:
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

<a id="First-Recommendations"></a>
### First Recommendations

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genres_list'])

def recommend(title, matrix, df, top_k=10, weight_genre=True):
    matches = df[df['title'].str.lower() == title.lower()]
    if matches.empty:
        raise ValueError(f"Book '{title}' not found in dataset.")
    pos = matches.index[0]

    scores = cosine_similarity(matrix[pos], matrix).flatten()
    scores[pos] = 0  # exclude self

    if weight_genre:
        input_vec = genre_matrix[pos]                  # shape (n_genres,)
        overlaps = genre_matrix.dot(input_vec)         # shape (n_books,)
        scores += 0.1 * overlaps
        scores[pos] = 0                                # re-zero self after boost

    top_idx = scores.argsort()[::-1][:top_k]
    results = df.iloc[top_idx][['title', 'author', 'genres_list']].copy()
    results['score'] = scores[top_idx]
    return results.reset_index(drop=True)

In [15]:
recommend("The Fall", tfidf_matrix, df)

,title,author,genres_list,score
0,The Myth of Sisyphus,Albert Camus,[Existentialism],0.249316
1,A Happy Death,Albert Camus,[Existentialism],0.165876
2,The Stranger,Albert Camus,"[Crime Fiction, Existentialism, Fiction, Absur...",0.162963
3,The First Man,Albert Camus,[Autobiographical novel],0.152216
4,The Plague,Albert Camus,"[Existentialism, Fiction, Absurdist fiction, N...",0.151790
5,The Outsider,Colin Wilson,"[Existentialism, Philosophy]",0.136384
6,Steppenwolf,Hermann Hesse,"[Existentialism, Speculative fiction, Fiction,...",0.132286
7,"They Shoot Horses, Don't They?",Horace McCoy,[Existentialism],0.122435
8,As I Walked Out One Midsummer Morning,Laurie Lee,[],0.072403
9,The Family of Pascual Duarte,Camilo José Cela,[Novel],0.071896


In [16]:
recommend("The Fall", tfidf_matrix, df, weight_genre=False)

,title,author,genres_list,score
0,The First Man,Albert Camus,[Autobiographical novel],0.152216
1,The Myth of Sisyphus,Albert Camus,[Existentialism],0.149316
2,As I Walked Out One Midsummer Morning,Laurie Lee,[],0.072403
3,The Family of Pascual Duarte,Camilo José Cela,[Novel],0.071896
4,Notes from Underground,Fedor Mikhaïlovitch Dostoïevski,[Novella],0.071288
5,Chuang Tse and the first emperor,Anna Russo,[Spirituality],0.067823
6,Veracity,NaN,"[Suspense, Dystopia]",0.066981
7,From an Abandoned Work,Samuel Beckett,[],0.066018
8,A Happy Death,Albert Camus,[Existentialism],0.065876
9,White Nights,Fyodor Dostoyevsky,[],0.065141


## Integrate OpenLibrary Data

This section generates three saved files: (refer to them as checkpoints)
- `ol_works_filtered.csv`
- `all_books_processed.csv`
- `final_books_dataset.csv`

If `final_books_dataset.csv` exists in the workspace, skip this entire section and jump to [Evaluation](#eval).

In [22]:
import os

def parse_works_dump_streaming(filepath, output_csv, min_desc_words=20):
    cols = ['ol_key', 'title', 'subjects', 'description', 'first_publish_date']
    
    kept = 0
    skipped = 0
    
    file_size = os.path.getsize(filepath)
    
    with open(filepath, 'r', encoding='utf-8', errors='replace') as fin, \
         open(output_csv, 'w', newline='', encoding='utf-8') as fout, \
         tqdm(total=file_size, unit='B', unit_scale=True, desc="Parsing OL dump") as pbar:
        
        writer = csv.DictWriter(fout, fieldnames=cols)
        writer.writeheader()
        
        for line in fin:
            pbar.update(len(line.encode('utf-8', errors='replace')))
            parts = line.split('\t', 4)
            if len(parts) < 5:
                skipped += 1
                continue
            try:
                data = json.loads(parts[4])
            except json.JSONDecodeError:
                skipped += 1
                continue
            
            title = data.get('title', '').strip()
            if not title:
                skipped += 1
                continue
            
            # Parse description
            desc = data.get('description', '')
            if isinstance(desc, dict):
                desc = desc.get('value', '')
            desc = (desc or '').strip()
            
            # Filter: must have description >= min_desc_words
            if len(desc.split()) < min_desc_words:
                skipped += 1
                continue
            
            subjects = data.get('subjects', [])
            
            writer.writerow({
                'ol_key': parts[1].strip(),
                'title': title,
                'subjects': json.dumps(subjects[:15]),
                'description': desc,
                'first_publish_date': data.get('first_publish_date', '')
            })
            kept += 1
    
    print(f"Done. Kept: {kept:,} | Skipped: {skipped:,}")
    return output_csv

In [23]:
parse_works_dump_streaming('ol_dump_works_latest.txt', 'ol_works_filtered.csv')

Parsing OL dump: 100%|██████████| 23.0G/23.0G [05:11<00:00, 73.8MB/s]

Done. Kept: 1,194,622 | Skipped: 39,523,625


'ol_works_filtered.csv'

In [17]:
def normalize_title(t):
    import re
    t = t.lower().strip()
    t = re.sub(r'^(the|a|an) ', '', t)
    t = re.sub(r'[^a-z0-9 ]', '', t)
    return t.strip()

In [18]:
ol_df = pd.read_csv('ol_works_filtered.csv')
ol_df['norm_title'] = ol_df['title'].apply(normalize_title)
ol_df = ol_df.drop_duplicates(subset='norm_title', keep='first')

df = pd.read_csv('booksummaries_processed.csv')
df['genres_list'] = df['genres_list'].apply(json.loads)
df['norm_title'] = df['title'].apply(normalize_title)

df = df.merge(
    ol_df[['norm_title', 'subjects', 'first_publish_date']].rename(
        columns={'subjects': 'ol_subjects'}),
    on='norm_title', how='left'
)

ol_only = ol_df[~ol_df['norm_title'].isin(df['norm_title'])].copy()
ol_only = ol_only.rename(columns={'description': 'summary', 'subjects': 'genres_list_raw'})
ol_only['genres_list'] = ol_only['genres_list_raw'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else []
)
ol_only['author'] = ''
ol_only['wiki_id'] = None
ol_only['freebase_id'] = None
ol_only['pub_date'] = ol_only['first_publish_date']
ol_only['genres'] = None
ol_only['ol_subjects'] = ol_only['genres_list'].apply(json.dumps)
ol_only['clean_summary'] = ol_only['summary'].apply(preprocess)
ol_only['combined'] = (ol_only['clean_summary'] + ' ' +
                       ol_only['genres_list'].apply(lambda g: ' '.join(g)))

shared_cols = [c for c in df.columns if c in ol_only.columns]
combined_df = pd.concat([df[shared_cols], ol_only[shared_cols]], ignore_index=True)
combined_df = combined_df.drop(columns=['norm_title'])

print(f"CMU books:      {len(df):,}")
print(f"OL-only books:  {len(ol_only):,}")
print(f"Combined total: {len(combined_df):,}")

combined_df.to_csv('all_books_processed.csv', index=False)

CMU books:      15,470
OL-only books:  999,472
Combined total: 1,014,942


In [19]:
print(f"Has wiki_id (CMU):     {combined_df['wiki_id'].notna().sum():,}")
print(f"OL-only (no wiki_id):  {combined_df['wiki_id'].isna().sum():,}")

print(f"Has clean_summary:     {combined_df['clean_summary'].notna().sum():,}")
print(f"Empty clean_summary:   {combined_df['clean_summary'].isna().sum():,}")

print(f"Has genres:            {combined_df['genres_list'].apply(lambda x: len(x) > 0).sum():,}")
print(f"No genres:             {combined_df['genres_list'].apply(lambda x: len(x) == 0).sum():,}")

# Summary length distribution
combined_df['summary_len'] = combined_df['clean_summary'].fillna('').apply(lambda x: len(x.split()))
print(combined_df['summary_len'].describe())

Has wiki_id (CMU):     15,470
OL-only (no wiki_id):  999,472
Has clean_summary:     1,014,942
Empty clean_summary:   0
Has genres:            916,156
No genres:             98,786
count    1.014942e+06
mean     6.971019e+01
std      9.710002e+01
min      0.000000e+00
25%      2.400000e+01
50%      5.300000e+01
75%      9.100000e+01
max      3.897600e+04
Name: summary_len, dtype: float64


In [20]:
# drop books with summaries longer than 500 words
print(f"Summaries longer than 500 words: {(combined_df['summary_len'] > 500).sum()}")

final_df = combined_df[combined_df['summary_len'] <= 500]
final_df.nlargest(100, 'summary_len')[['title', 'author', 'summary_len']]

final_df.to_csv('final_books_dataset.csv', index=False)

Summaries longer than 500 words: 3254


In [22]:
print(len(final_df))

1011688


<a id="eval"></a>
# Evaluation

1. TF-IDF
2. Embeddings + FAISS
3. Recommendations

In [ ]:
import faiss
import numpy as np
import pandas as pd
import ast
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
final_df = pd.read_csv('final_books_dataset.csv', low_memory=False, dtype={
    'freebase_id': 'str',
    'author': 'str',
    'genres': 'str'})
final_df['genres_list'] = final_df['genres_list'].apply(lambda x: ast.literal_eval(x))

final_df['combined'] = (
    final_df['clean_summary'].fillna('') + ' ' +
    final_df['author'].fillna('') + ' ' +
    final_df['genres_list'].apply(lambda g: ' '.join(g) if isinstance(g, list) else '')
)

print(f"Total books: {len(final_df):,}")

Total books: 1,011,688


In [8]:
full_tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 1))
full_tfidf_matrix = full_tfidf.fit_transform(final_df['combined'])

In [9]:
joblib.dump(full_tfidf_matrix, 'full_tfidf_matrix.pkl')
joblib.dump(full_tfidf, 'full_tfidf_vectorizer.pkl')

['full_tfidf_vectorizer.pkl']

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(sparse_output=True)
full_genre_matrix = mlb.fit_transform(final_df['genres_list'])

In [6]:
joblib.dump(full_genre_matrix, 'full_genre_matrix.pkl')

['full_genre_matrix.pkl']

In [ ]:
# full_tfidf_matrix = joblib.load('full_tfidf_matrix.pkl')
# full_tfidf = joblib.load('full_tfidf_vectorizer.pkl')
# full_genre_matrix = joblib.load('full_genre_matrix.pkl')

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using: {device}")

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
batch_size = 512 if device == 'cuda' else 64

full_embeddings = model.encode(
    final_df['clean_summary'].fillna('').tolist(),
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

faiss.normalize_L2(full_embeddings)
full_index = faiss.IndexFlatIP(full_embeddings.shape[1])
full_index.add(full_embeddings)

faiss.write_index(full_index, 'full_book_index.faiss')
np.save('full_embeddings.npy', full_embeddings)

In [ ]:
# full_index = faiss.read_index('full_book_index.faiss')
# full_embeddings = np.load('full_embeddings.npy')

In [ ]:
def recommend_tfidf_full(title, top_k=10, weight_genre=True):
    matches = final_df[final_df['title'].str.lower() == title.lower()]
    if matches.empty:
        raise ValueError(f"'{title}' not found.")
    pos = matches.index[0]

    scores = cosine_similarity(full_tfidf_matrix[pos], full_tfidf_matrix).flatten()
    scores[pos] = 0

    if weight_genre:
        overlaps = np.asarray(full_genre_matrix.dot(full_genre_matrix[pos].T).todense()).flatten()
        scores += 0.1 * overlaps
        scores[pos] = 0

    top_idx = scores.argsort()[::-1][:top_k]
    results = final_df.iloc[top_idx][['title', 'author', 'genres_list']].copy()
    results['score'] = scores[top_idx]
    return results.reset_index(drop=True)


def recommend_semantic_full(title, top_k=10):
    matches = final_df[final_df['title'].str.lower() == title.lower()]
    if matches.empty:
        raise ValueError(f"'{title}' not found.")
    pos = matches.index[0]

    query = full_embeddings[pos:pos+1]
    scores, indices = full_index.search(query, top_k + 1)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == pos:
            continue
        results.append({
            'title': final_df.iloc[idx]['title'],
            'author': final_df.iloc[idx]['author'],
            'genres': final_df.iloc[idx]['genres_list'],
            'score': float(score)
        })
    return pd.DataFrame(results[:top_k])


def recommend_personalized_full(liked_titles, theme_keywords=None, top_k=10):
    liked_idxs = [
        final_df[final_df['title'].str.lower() == t.lower()].index[0]
        for t in liked_titles
        if not final_df[final_df['title'].str.lower() == t.lower()].empty
    ]
    if not liked_idxs:
        raise ValueError("None of the titles found.")

    user_vec = full_embeddings[liked_idxs].mean(axis=0, keepdims=True).copy()

    if theme_keywords:
        theme_vec = model.encode([' '.join(theme_keywords)],
                                  convert_to_numpy=True).astype('float32')
        user_vec = 0.7 * user_vec + 0.3 * theme_vec

    faiss.normalize_L2(user_vec)
    scores, indices = full_index.search(user_vec.astype('float32'), top_k + len(liked_idxs))

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if final_df.iloc[idx]['title'] in liked_titles:
            continue
        results.append({
            'title': final_df.iloc[idx]['title'],
            'author': final_df.iloc[idx]['author'],
            'genres': final_df.iloc[idx]['genres_list'],
            'score': float(score)
        })
    return pd.DataFrame(results[:top_k])

In [ ]:
def run_eval(test_books, recommend_fn, label):
    rows = []
    for title in test_books:
        recs = recommend_fn(title)
        for rank, row in enumerate(recs.itertuples(), 1):
            rows.append({
                'input': title,
                'rank': rank,
                'recommended': getattr(row, 'title', ''),
                'score': getattr(row, 'score', 0),
                'relevant': None,
                'rating': None
            })

    out = pd.DataFrame(rows)
    out.to_csv(f'eval_{label}.csv', index=False)
    print(f"Saved eval_{label}.csv")

def compute_metrics(csv_path, k=10):
    df_eval = pd.read_csv(csv_path)
    results = {}
    for title, group in df_eval.groupby('input'):
        judgments = group.head(k)
        p_at_k = judgments['relevant'].sum() / k
        gains = judgments['rating'].fillna(0).tolist()
        dcg  = sum(g / np.log2(i+2) for i, g in enumerate(gains))
        ideal = sorted(gains, reverse=True)
        idcg = sum(g / np.log2(i+2) for i, g in enumerate(ideal))
        ndcg = dcg / idcg if idcg else 0
        results[title] = {'P@10': round(p_at_k, 3), 'NDCG@10': round(ndcg, 3)}
    return pd.DataFrame(results).T

In [ ]:
test_books = [
    "The Fall",
    "Dune", 
    "Pride and Prejudice",
    "The Great Gatsby",
    "Harry Potter and the Sorcerer's Stone"
]

In [ ]:
run_eval(recommend_tfidf_full, 'tfidf')
run_eval(recommend_semantic_full, 'semantic')

Manual annotation of relevance (0/1) and rating (scale of 1-5) for each of the top 10 recommended books is completed here.

In [ ]:
# print(compute_metrics('eval_tfidf.csv'))
# print(compute_metrics('eval_semantic.csv'))